# 13 Normal MRT and Qualification Pilot

Controlled TRAIN-only pilot for six candidate Normal control periods. This notebook documents the small RouteViews download, parser inspection, 5-minute window construction, behavioral feature extraction, and rule-based Normal qualification result.

Scope guardrails: no Validation or Holdout data, no final Normal labels, no CAIDA/RPKI features, no model training, no ablation, and no changes to the frozen v2 roster or partitions.

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve()
PILOT = ROOT / 'data' / 'pilot'

periods = pd.read_csv(PILOT / 'normal_mrt_pilot_periods_v1.csv')
manifest = pd.read_csv(PILOT / 'normal_mrt_pilot_download_manifest_v1.csv')
download_audit = pd.read_csv(PILOT / 'normal_mrt_pilot_download_audit_v1.csv')
raw_counts = pd.read_csv(PILOT / 'normal_pilot_raw_bgp_counts_v1.csv')
relevance = pd.read_csv(PILOT / 'normal_pilot_relevance_summary_v1.csv')
window_counts = pd.read_csv(PILOT / 'normal_pilot_window_counts_v1.csv')
features = pd.read_parquet(PILOT / 'normal_pilot_behavioral_features_v1.parquet')
qualification = pd.read_csv(PILOT / 'normal_pilot_qualification_results_v1.csv')
comparison = pd.read_csv(PILOT / 'normal_vs_incident_behavior_summary_v1.csv')

## Selected TRAIN Normal Periods

In [ ]:
assert len(periods) == 6
assert set(periods['partition']) == {'train'}
assert periods['associated_category'].value_counts().to_dict() == {'Hijack': 2, 'Leak': 2, 'Outage': 2}
periods

## Download Manifest and Audit

In [ ]:
summary = {
    'planned_mrt_files': len(manifest),
    'remote_available': int(manifest['remote_available'].sum()),
    'estimated_mib': round(manifest['estimated_content_length'].fillna(0).sum() / 1024 / 1024, 2),
    'downloaded_files': int(download_audit['downloaded_files'].sum()),
    'audit_statuses': download_audit['status'].value_counts().to_dict(),
}
summary

In [ ]:
download_audit

## Raw BGP Counts and Relevance

In [ ]:
raw_counts

In [ ]:
relevance

## 5-Minute Windows and Behavioral Features

In [ ]:
assert len(features) == len(window_counts)
assert features['window_id'].is_unique
assert set(features['partition']) == {'train'}
{
    'candidate_windows': len(features),
    'feature_columns': features.shape[1],
    'windows_with_relevant_observations': int(features['has_relevant_observation'].sum()),
    'windows_without_relevant_observations': int((~features['has_relevant_observation']).sum()),
}

## Normal Qualification Result

In [ ]:
qualification['final_state'].value_counts()

In [ ]:
qualification.merge(
    periods[['candidate_period_id', 'associated_category', 'associated_incident_uid', 'candidate_offset']],
    on='candidate_period_id',
).groupby(['associated_category', 'associated_incident_uid', 'candidate_offset'])['final_state'].value_counts().unstack(fill_value=0)

## Descriptive Comparison

No classifier is trained here. Because no windows passed the tightened Normal qualification, the comparison table intentionally contains empty Normal columns and should not be used for threshold tuning.

In [ ]:
comparison.head(10)

## Pilot Verdict

The RouteViews download and parser are ready for this workflow, but the current Normal candidate selection/qualification method is not ready to scale as-is. A trustworthy Normal window must have directly relevant RouteViews evidence for the associated ASN/prefix/network and a sustained stable block. In this six-period pilot, only 4 of 63 windows had direct relevant observations and none satisfied the full sustained-stability policy, so all candidate windows remain `UNKNOWN`.